# cross-entropy-classification-loss — ex2: cross-entropy with ignore_index for masked padding labels

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `cross-entropy-classification-loss`. Running the final beacon cell reports progress against the `Loss: Cross-entropy classification` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Loss: Cross-entropy classification` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cross-entropy-classification-loss`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cross-entropy-classification-loss"
DD_SUBTOPIC = "Loss: Cross-entropy classification"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Cross-entropy with `ignore_index` — quick refresher

Language modeling and any sequence task pad short examples to a common length. The padding positions have a placeholder label (e.g. `-100` by convention) that the loss MUST skip — including them would let the model 'win' by predicting the pad token everywhere.

`F.cross_entropy` has a built-in `ignore_index` argument that filters labels matching the sentinel BEFORE averaging:
```python
loss = F.cross_entropy(logits, labels, ignore_index=-100)
# equivalent to:
mask = labels != -100
loss = F.cross_entropy(logits[mask], labels[mask])
```
The two forms are NUMERICALLY equivalent (mean over the kept examples only). The built-in arg saves the mask + index step.

### Exercise 2 — cross-entropy with ignore_index for masked padding labels

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `ignore_index` filtering to cross-entropy so padded label positions don't contribute to the loss, and verify the masked result equals the masked-out hand-computed loss.
> Keywords: cross-entropy, ignore-index, padding-mask, language-modeling
> ```

**KCs targeted:** `cross-entropy-takes-logits-not-probs`, `cross-entropy-ignore-index-skips-padding`

Implement `ex2_masked_cross_entropy(logits, labels, pad_id)`. The cross-entropy loss with padding positions masked out.

Two acceptable implementations (both must give the same answer):
1. **Built-in route**: `return F.cross_entropy(logits, labels, ignore_index=pad_id)`.
2. **Manual route**: `mask = labels != pad_id; return F.cross_entropy(logits[mask], labels[mask])`.

You can use either. The test verifies your output matches the manual reference EXACTLY (within 1e-5).

Inputs:
- `logits`: `(N, C)` float tensor.
- `labels`: `(N,)` int64 — possibly containing `pad_id` (default -100) as a 'skip me' sentinel.
- `pad_id`: int — labels equal to this are NOT scored.

Output: scalar loss tensor.

**Critical:** the loss must AVERAGE over the kept (non-padded) positions only, not over the total length. The test gives you a 16-token batch with 6 padded positions; the correct denominator is 10, not 16.

In [ ]:
import torch.nn.functional as F


def ex2_masked_cross_entropy(logits, labels, pad_id=-100):
    return F.cross_entropy(logits, labels, ignore_index=pad_id)


<details><summary>Solution</summary>

```python
import torch.nn.functional as F


def ex2_masked_cross_entropy(logits, labels, pad_id=-100):
    return F.cross_entropy(logits, labels, ignore_index=pad_id)
```

**Why -100 by convention.** PyTorch's `ignore_index` default is `-100` specifically because no valid class index is negative — you can't accidentally collide with a real label. HuggingFace tokenizers honor this convention: padded-token labels are set to -100 in their default data collators.

**`ignore_index` vs `weight`.** `weight=t.tensor([...])` re-weights by class (e.g. for class imbalance); `ignore_index` removes specific POSITIONS entirely. Use both together for imbalanced-with-padding tasks like NER.

**Where this lives in ARENA chap-3.** The transformer training loss is `F.cross_entropy(logits.view(-1, V), labels.view(-1), ignore_index=tokenizer.pad_token_id)` — exact same pattern, flattened across batch × seq.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()